# Earthquake Watch - my LangChain agent

**Problem:** Almaty is in a seismic zone. When there is an earthquake, people want to know fast:
*where was it, how strong, could I feel it, what should I do?*
ChatGPT can't answer this, because it does not have live data.

**My agent** uses real data from the **USGS** earthquake API and some simple tools I wrote:

| Tool | What it does |
|---|---|
| `get_city_coordinates` | finds latitude/longitude of a city (Open-Meteo Geocoding API) |
| `get_recent_earthquakes` | gets real earthquakes near a point (USGS API) and calculates distance |
| `explain_magnitude` | explains what a magnitude means and if people can feel it at some distance |
| `make_emergency_kit` | makes an emergency kit list for N people and N days (exact math) |
| `save_note` | saves a note to a text file |

**LLM (the brain):** Claude Haiku 5.5 (`claude-haiku-5-5`) through `langchain-anthropic`.

**LangChain parts:** `ChatPromptTemplate`, LCEL chain (`prompt | llm | parser`), Pydantic parser,
agent (`create_tool_calling_agent` + `AgentExecutor`), memory (`RunnableWithMessageHistory`), LangSmith tracing.

## 1. Install libraries

In [4]:
!pip install -q langchain-classic langchain-anthropic langsmith requests python-dotenv

## 2. API keys

I don't write keys in the code. In Colab I put them in **Secrets** (key icon on the left):
- `ANTHROPIC_API_KEY` - from console.anthropic.com
- `LANGSMITH_API_KEY` - from smith.langchain.com (free)

If I run it on my laptop, the keys are in a `.env` file (and `.env` is in `.gitignore`).

In [5]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "earthquake-watch"

print("Anthropic key found:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("LangSmith key found:", bool(os.environ.get("LANGSMITH_API_KEY")))

Anthropic key found: True
LangSmith key found: True


## 3. The LLM

I use Claude Haiku 5.5. It is the fast and cheap Claude model, and it is still good at choosing tools
and following the system prompt.
`max_tokens` is the limit for one answer.

In [6]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-haiku-5-5", max_tokens=4096)

# quick test
response = llm.invoke("Say hello in one short sentence.")
print(response.text)

Hello, it's nice to meet you!


## 4. Tools (the "hands" of the agent)

Rules I followed for every tool:
- clear **name**, **docstring** and **argument types** - the LLM reads them to choose the tool
- **no crash**: if the API is down or the city is not found, the tool returns an error message as text

In [7]:
import math
import requests
from datetime import datetime, timedelta, timezone
from langchain_core.tools import tool


def haversine_km(lat1, lon1, lat2, lon2):
    """Distance between two points on Earth (haversine formula). Normal function, not a tool."""
    R = 6371  # radius of Earth in km
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return round(2 * R * math.asin(math.sqrt(a)))


@tool
def get_city_coordinates(city: str) -> str:
    """Find the latitude and longitude of a city by its name (for example 'Almaty' or 'Tokyo').
    Use this first when the user talks about a city, because the earthquake tool needs coordinates."""
    try:
        url = "https://geocoding-api.open-meteo.com/v1/search"
        r = requests.get(url, params={"name": city, "count": 1}, timeout=10)
        data = r.json()
    except Exception as e:
        return f"Error: geocoding service is not available ({e})"

    if "results" not in data:
        return f"Error: city '{city}' was not found. Ask the user to check the spelling."

    place = data["results"][0]
    return (f"{place['name']}, {place.get('country', '')}: "
            f"latitude={place['latitude']}, longitude={place['longitude']}")


@tool
def get_recent_earthquakes(latitude: float, longitude: float, days: int = 7,
                           radius_km: int = 500, min_magnitude: float = 2.5) -> str:
    """Get REAL earthquakes from the USGS database near a point.
    Args: latitude and longitude of the point, days = how many days back to search,
    radius_km = search radius in km, min_magnitude = only earthquakes stronger than this.
    Returns a list of earthquakes with magnitude, place, time (UTC), depth and distance from the point."""
    start_date = (datetime.now(timezone.utc) - timedelta(days=days)).strftime("%Y-%m-%d")
    params = {
        "format": "geojson",
        "latitude": latitude,
        "longitude": longitude,
        "maxradiuskm": radius_km,
        "starttime": start_date,
        "minmagnitude": min_magnitude,
        "orderby": "time",
        "limit": 15,
    }
    try:
        r = requests.get("https://earthquake.usgs.gov/fdsnws/event/1/query", params=params, timeout=15)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        return f"Error: USGS earthquake service is not available right now ({e})"

    quakes = data["features"]
    if len(quakes) == 0:
        return f"No earthquakes of magnitude {min_magnitude}+ within {radius_km} km in the last {days} days."

    lines = [f"Found {len(quakes)} earthquakes (magnitude {min_magnitude}+, {radius_km} km, last {days} days):"]
    for q in quakes:
        info = q["properties"]
        lon, lat, depth = q["geometry"]["coordinates"]
        time = datetime.fromtimestamp(info["time"] / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
        dist = haversine_km(latitude, longitude, lat, lon)
        lines.append(f"- M{info['mag']} | {info['place']} | {time} | depth {depth} km | {dist} km from the point")
    return "\n".join(lines)


@tool
def explain_magnitude(magnitude: float, distance_km: float) -> str:
    """Explain what an earthquake magnitude means and estimate if people can feel it
    at a given distance (in km) from the epicenter. Use it when the user asks 'is it dangerous?'
    or 'could I feel it?'."""
    if magnitude < 3:
        level, felt_radius = "micro/minor: usually not felt", 20
    elif magnitude < 4:
        level, felt_radius = "minor: felt by some people, no damage", 60
    elif magnitude < 5:
        level, felt_radius = "light: felt by many people, objects shake, damage is rare", 150
    elif magnitude < 6:
        level, felt_radius = "moderate: can damage weak buildings", 300
    elif magnitude < 7:
        level, felt_radius = "strong: can cause damage in populated areas", 600
    else:
        level, felt_radius = "major: serious damage over large areas", 1000

    if distance_km <= felt_radius:
        felt = f"At {distance_km} km people will PROBABLY feel it (felt up to ~{felt_radius} km)."
    else:
        felt = f"At {distance_km} km people will PROBABLY NOT feel it (felt up to ~{felt_radius} km)."
    return f"Magnitude {magnitude} = {level}. {felt} This is a rough estimate, depth and soil also matter."


@tool
def make_emergency_kit(people: int, days: int = 3) -> str:
    """Make an emergency kit (go-bag) checklist with exact amounts of water and food
    for a number of people and days."""
    if people < 1 or days < 1:
        return "Error: people and days must be at least 1."
    water = 3 * people * days      # 3 liters per person per day
    meals = 3 * people * days      # 3 meals per person per day
    items = [
        f"Water: {water} liters (3 L per person per day)",
        f"Food that does not need cooking: {meals} meals (canned food, bars, nuts)",
        "First aid kit + personal medicines",
        "Flashlight + extra batteries, power bank",
        "Copies of documents (ID, passports) in a plastic bag",
        "Some cash",
        "Warm clothes and a blanket for each person",
        "Whistle (to call for help)",
    ]
    return f"Emergency kit for {people} people for {days} days:\n- " + "\n- ".join(items)


@tool
def save_note(text: str) -> str:
    """Save a note (for example an emergency plan or a kit list) to the file my_notes.txt.
    Use it only when the user asks to save or remember something in a file."""
    try:
        with open("my_notes.txt", "a", encoding="utf-8") as f:
            f.write(f"--- {datetime.now().strftime('%Y-%m-%d %H:%M')} ---\n{text}\n\n")
        return "Saved to my_notes.txt"
    except Exception as e:
        return f"Error: could not save the note ({e})"


tools = [get_city_coordinates, get_recent_earthquakes, explain_magnitude, make_emergency_kit, save_note]

## 5. LCEL chain with structured output (`prompt | llm | parser`)

This chain gets the raw earthquake text from the USGS tool and turns it into a
**Pydantic object** - a short risk report with fixed fields.
`PydanticOutputParser` tells the LLM what JSON format to use and then checks the answer.

In [8]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


class EarthquakeReport(BaseModel):
    city: str = Field(description="city name")
    total_earthquakes: int = Field(description="how many earthquakes were found")
    strongest_magnitude: float = Field(description="magnitude of the strongest earthquake, 0 if none")
    strongest_place: str = Field(description="place of the strongest earthquake, empty if none")
    risk_level: Literal["low", "medium", "high"] = Field(description="risk for people in the city right now")
    advice: list[str] = Field(description="2-3 short practical tips")


parser = PydanticOutputParser(pydantic_object=EarthquakeReport)

report_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a seismic safety analyst. You only use the earthquake data you are given, "
     "you never invent earthquakes. Risk level: 'low' if nothing stronger than M4.5 within 150 km, "
     "'high' if M6+ within 300 km, otherwise 'medium'.\n{format_instructions}"),
    ("human", "City: {city}\n\nEarthquake data from USGS:\n{earthquake_data}"),
])

report_chain = report_prompt | llm | parser

# run it: first get real data with the tool, then give it to the chain
data = get_recent_earthquakes.invoke({"latitude": 43.25, "longitude": 76.91, "days": 30})
report = report_chain.invoke({
    "city": "Almaty",
    "earthquake_data": data,
    "format_instructions": parser.get_format_instructions(),
})

print(type(report))
print("City:", report.city)
print("Earthquakes:", report.total_earthquakes)
print("Strongest:", report.strongest_magnitude, "-", report.strongest_place)
print("Risk:", report.risk_level)
for tip in report.advice:
    print(" *", tip)

<class '__main__.EarthquakeReport'>
City: Almaty
Earthquakes: 5
Strongest: 4.9 - 39 km NW of Kuqa, China
Risk: low
 * No significant nearby activity: all recorded quakes were moderate and 333+ km away, so no immediate action is needed.
 * Almaty is in a seismically active region, so secure heavy furniture and keep an emergency kit with water, a flashlight and first aid supplies.
 * Review 'Drop, Cover, and Hold On' with your household and know the safe spots in your home and workplace.


## 6. The agent

**System prompt** = role + rules + answer style.
**User prompt template** = `{input}` (the question).
`chat_history` = memory (old messages), `agent_scratchpad` = tool calls and tool results of the current question.

How the agent works step by step:
1. The prompt (system + history + question + tool list) goes to Claude.
2. Claude reads the tool names and docstrings and decides: answer now, or call a tool (it returns a *tool call* with arguments).
3. `AgentExecutor` runs the tool and puts the result into `agent_scratchpad`.
4. Claude gets the result and decides again (maybe call another tool).
5. When Claude answers without a tool call -> this is the final answer.

In [9]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

SYSTEM_PROMPT = """You are "Earthquake Watch", a calm and friendly assistant who helps people in Kazakhstan
and Central Asia understand earthquakes and stay safe.

Rules:
1. For any question about real earthquakes, ALWAYS use the tools to get real data from USGS.
   Never invent earthquakes, magnitudes, times or distances.
2. If you only have a city name, first call get_city_coordinates, then get_recent_earthquakes.
3. If a tool returns an error, tell the user honestly and suggest what to do.
4. Times from USGS are in UTC. Kazakhstan time is UTC+5, convert it when it helps the user.
5. Use explain_magnitude when the user asks if an earthquake is dangerous or if they could feel it.
6. Use make_emergency_kit for kit/go-bag questions, so the amounts are exact.
7. Use save_note only when the user asks to save something.
8. You are not an official emergency service. In a real emergency tell people to call 112.

Answer style:
- Answer in the same language as the user.
- Short and clear: a few sentences or a short list, no long texts.
- Show the important numbers (magnitude, distance, time).
- End with one short practical safety tip when it makes sense."""

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)
# return_intermediate_steps=True -> I can print which tools the agent used
agent_executor = AgentExecutor(agent=agent, tools=tools, max_iterations=8, return_intermediate_steps=True)

## 7. Memory

`RunnableWithMessageHistory` saves every question and answer in a chat history (one history per `session_id`).
Before each new question it puts the old messages into `chat_history`, so the agent understands follow-up questions
like *"what about the strongest one?"*.

Small detail: Claude's answer can come as a list of blocks (text + "thinking" block), so I keep only the text.

In [10]:
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory


def only_text(content):
    """Claude can return a list of blocks. Keep only the text parts."""
    if isinstance(content, str):
        return content
    texts = []
    for block in content:
        if isinstance(block, dict) and block.get("type") == "text":
            texts.append(block["text"])
    return "".join(texts)


def clean_result(result):
    """AgentExecutor returns input, output and intermediate_steps. I keep clean text + the steps."""
    return {"output": only_text(result["output"]), "steps": result["intermediate_steps"]}


agent_with_clean_output = agent_executor | RunnableLambda(clean_result)

# all chat histories: {session_id: history}
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


agent_with_memory = RunnableWithMessageHistory(
    agent_with_clean_output,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="output",
)


def ask(question, session_id="default"):
    result = agent_with_memory.invoke(
        {"input": question},
        config={"configurable": {"session_id": session_id}},
    )
    print("USER:", question)
    for action, tool_result in result["steps"]:
        print(f"  [tool] {action.tool}({action.tool_input})")
    print("AGENT:", result["output"])
    print()

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 8. Example conversations

### Conversation 1 - several messages (follow-up questions use memory)

In [11]:
ask("Were there any earthquakes near Almaty in the last 7 days?", session_id="conversation_1")

/tmp/ipykernel_1746/724883627.py:29: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


USER: Were there any earthquakes near Almaty in the last 7 days?
  [tool] get_city_coordinates({'city': 'Almaty'})
  [tool] get_recent_earthquakes({'latitude': 43.25249, 'longitude': 76.9115, 'days': 7})
AGENT: Yes, but there were only **2 earthquakes**, and both were far from Almaty. I searched within 500 km for magnitude 2.5 and above.

| Magnitude | Location | Time (Almaty, UTC+5) | Depth | Distance from Almaty |
|---|---|---|---|---|
| **M4.9** | 39 km NW of Kuqa, China | 23 Nov, 07:14 (02:14 UTC) | 10 km | ~484 km |
| **M4.2** | 34 km NW of Kuqa, China | 23 Nov, 13:10 (08:10 UTC) | 10 km | ~491 km |

Both were near the far edge of the search area. Earthquakes this size that far away usually can't be felt in Almaty, and there were none close to the city. If you'd like, I can check whether these two could have been felt.

**Tip:** Almaty is in an active earthquake zone, so make sure your heavy furniture and shelves are fastened to the walls.



In [12]:
ask("What was the strongest one? Could people in Almaty feel it?", session_id="conversation_1")

USER: What was the strongest one? Could people in Almaty feel it?
  [tool] get_city_coordinates({'city': 'Almaty'})
  [tool] get_recent_earthquakes({'latitude': 43.25249, 'longitude': 76.9115, 'days': 7})
  [tool] explain_magnitude({'magnitude': 4.9, 'distance_km': 484})
AGENT: The strongest one was the **M4.9** earthquake **39 km NW of Kuqa, China**, on **23 Nov at 07:14 Almaty time** (02:14 UTC). It was 10 km deep.

**People in Almaty probably didn't feel it.**
- An M4.9 earthquake is "light." People near the center feel it and objects shake, but damage is rare.
- It's usually felt up to about **150 km** away, and this one was about **484 km** from Almaty.
- This is only a rough estimate, because depth and soil type also matter.

**Tip:** Know the safe spots in your home, like under a sturdy table or next to an inner wall, so you can "Drop, Cover, Hold On" right away when shaking starts.



In [13]:
ask("Ok, I'm worried. Make an emergency kit list for my family of 4 people for 3 days and save it to my notes.",
    session_id="conversation_1")

USER: Ok, I'm worried. Make an emergency kit list for my family of 4 people for 3 days and save it to my notes.
  [tool] make_emergency_kit({'people': 4, 'days': 3})
  [tool] save_note({'text': 'EMERGENCY KIT (GO-BAG) - Family of 4, 3 days\n- Water: 36 liters (3 L per person per day)\n- Food that does not need cooking: 36 meals (canned food, bars, nuts)\n- First aid kit + personal medicines\n- Flashlight + extra batteries, power bank\n- Copies of documents (ID, passports) in a plastic bag\n- Some cash\n- Warm clothes and a blanket for each person\n- Whistle (to call for help)\nEmergency number in Kazakhstan: 112'})
AGENT: It's normal to feel worried. Making a kit is a good way to get ready. Here's the list for **4 people for 3 days**, and it's **saved to my_notes.txt**:

- **Water: 36 liters** (3 L per person per day)
- **Food: 36 meals** that don't need cooking, like canned food, bars and nuts
- First aid kit and any medicines your family takes
- Flashlight, extra batteries and a powe

In [14]:
# the note really is in the file
print(open("my_notes.txt", encoding="utf-8").read())

--- 2026-11-24 17:29 ---
EMERGENCY KIT (GO-BAG) - Family of 4, 3 days
- Water: 36 liters (3 L per person per day)
- Food that does not need cooking: 36 meals (canned food, bars, nuts)
- First aid kit + personal medicines
- Flashlight + extra batteries, power bank
- Copies of documents (ID, passports) in a plastic bag
- Some cash
- Warm clothes and a blanket for each person
- Whistle (to call for help)
Emergency number in Kazakhstan: 112




### Conversation 2 - another city, stronger earthquakes

In [15]:
ask("Show me earthquakes of magnitude 4.5 or more near Tokyo in the last 14 days.", session_id="conversation_2")

USER: Show me earthquakes of magnitude 4.5 or more near Tokyo in the last 14 days.
  [tool] get_city_coordinates({'city': 'Tokyo'})
  [tool] get_recent_earthquakes({'latitude': 35.6895, 'longitude': 139.69171, 'days': 14, 'min_magnitude': 4.5, 'radius_km': 500})
AGENT: I found **8 earthquakes of magnitude 4.5 or higher** within 500 km of Tokyo in the last 14 days, from USGS data. Times are in UTC, with Tokyo time (UTC+9) in brackets:

| Mag | Location | Time UTC (Tokyo) | Depth | Distance from Tokyo |
|---|---|---|---|---|
| **5.4** | 20 km NW of Anamizu | Nov 16, 04:24 (13:24) | 274 km | 323 km |
| **5.2** | 112 km SSE of Tateyama | Nov 21, 10:09 (19:09) | 10 km | 193 km |
| **5.2** | 86 km NW of Ryōtsu-minato | Nov 18, 09:37 (18:37) | 10 km | 373 km |
| 4.9 | 83 km ESE of Takahagi | Nov 10, 05:20 (14:20) | 35 km | 181 km |
| 4.6 | 68 km SSE of Tateyama | Nov 21, 17:30 (Nov 22, 02:30) | 65 km | 148 km |
| 4.6 | 130 km ENE of Hasaki | Nov 11, 03:56 (12:56) | 10 km | 230 km |
| 4.6 | 89

### Conversation 3 - question in Russian + a city that does not exist (error handling)

In [16]:
ask("Были ли землетрясения возле города Qwertyuiopcity?", session_id="conversation_3")

USER: Были ли землетрясения возле города Qwertyuiopcity?
  [tool] get_city_coordinates({'city': 'Qwertyuiopcity'})
AGENT: Не удалось найти город **«Qwertyuiopcity»**: поиск координат не дал результата. Поэтому проверить землетрясения рядом с ним я пока не могу.

Что можно сделать:
- **Проверьте написание.** Можно написать название по-русски или по-английски, например «Алматы» / «Almaty».
- **Укажите ближайший крупный город**, например Алматы, Шымкент, Талдыкорган или Бишкек.
- **Пришлите координаты** (широту и долготу). По ним я сразу проверю данные USGS.

💡 **Совет:** заранее найдите в доме безопасное место: под прочным столом или у несущей стены, подальше от окон.



In [17]:
ask("Извини, я имел в виду Бишкек. Опасно ли землетрясение 5.5 на расстоянии 200 км?", session_id="conversation_3")

USER: Извини, я имел в виду Бишкек. Опасно ли землетрясение 5.5 на расстоянии 200 км?
  [tool] get_city_coordinates({'city': 'Bishkek'})
  [tool] explain_magnitude({'magnitude': 5.5, 'distance_km': 200})
  [tool] get_recent_earthquakes({'latitude': 42.87, 'longitude': 74.59, 'days': 7, 'radius_km': 500, 'min_magnitude': 2.5})
AGENT: Землетрясение **магнитудой 5.5 на расстоянии 200 км** для Бишкека обычно **не опасно**, но его, скорее всего, почувствуют:
- Это умеренное землетрясение. Рядом с эпицентром оно может повредить слабые здания.
- На расстоянии 200 км толчки обычно ощущаются (их чувствуют примерно до 300 км), но разрушения маловероятны.
- Это приблизительная оценка: сила толчков зависит ещё от глубины очага и грунта.

**Землетрясения около Бишкека за последние 7 дней** (данные USGS, радиус 500 км, магнитуда от 2.5):
- **M5.3**, 7 км к юго-востоку от Караканджи (Таджикистан)
- 18.11.2026 в 00:46 UTC (05:46 по времени Казахстана)
- Глубина 10 км, **480 км от Бишкека**. В городе е

### Chat with the agent yourself

Run this cell and type your own questions in the box. The agent remembers the whole chat.
Type `exit` to stop.

Ideas: *"Any earthquakes near Shymkent this month?"*, *"Is M6 dangerous 400 km away?"*,
*"Make a kit for 2 people for 5 days"*.

In [18]:
print("Write your question. Write 'exit' to stop.")
while True:
    question = input("You: ")
    if question.strip().lower() in ["exit", "quit", "stop", "выход"]:
        print("Bye!")
        break
    if question.strip() == "":
        continue
    ask(question, session_id="my_chat")

Write your question. Write 'exit' to stop.
You: Было ли в Алмате землетересение
USER: Было ли в Алмате землетересение
  [tool] get_city_coordinates({'city': 'Almaty'})
  [tool] get_recent_earthquakes({'latitude': 43.25249, 'longitude': 76.9115, 'days': 7, 'radius_km': 300})
  [tool] get_recent_earthquakes({'latitude': 43.25249, 'longitude': 76.9115, 'days': 30, 'radius_km': 300})
AGENT: Нет. По данным USGS, за **последние 30 дней** в радиусе **300 км** от Алматы не было землетрясений магнитудой **2.5 и выше**.

Это значит, что сильных и заметных толчков рядом с городом не было. Совсем слабые толчки (меньше 2.5) могли быть, но в этом поиске их нет. Люди их обычно почти не чувствуют.

Могу проверить период подольше, больший радиус или более слабые толчки, если нужно.

💡 **Совет:** Алматы находится в сейсмоопасной зоне, поэтому лучше заранее закрепить высокую мебель и держать дома тревожный чемоданчик. В экстренной ситуации звоните **112**.

You: Есть ли какие то доготки про след землетре

## 9. LangSmith traces

Tracing is turned on in step 2 (`LANGSMITH_TRACING=true`). All runs go to the project **earthquake-watch**
at https://smith.langchain.com.

In a trace I can see every step: the prompt, which tool Claude chose, the arguments, the tool result and the final answer.

*(Screenshots are in the `screenshots/` folder and in README.md.)*